In [1]:
import pandas as pd
from sklearn.metrics import f1_score
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.utils import compute_sample_weight

---
# Modèle MLPClassifier

---
## Validation croisée

In [ ]:
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

# les 32 colonnes d'entreprise
company_cols = [f'company_feature_{i}' for i in range(32)]

#regroupe sous un même identifiant les lignes qui ont la meme entreprise
df_x_clean['company_group'] = df_x_clean[company_cols].apply(lambda row: hash(tuple(row)), axis=1)
groups_company = df_x_clean['company_group']

X = df_x_clean.drop(columns=['id', 'company_group'])
y = df_y_clean['target']

cv = StratifiedGroupKFold(n_splits=5,random_state=42, shuffle=True)
scores_f1 = []

#entraînement et d'évaluation pli par pli
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups=groups_company)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    model = MLPClassifier(
        random_state=42,
        early_stopping=True,
        max_iter=300, 
        learning_rate_init=0.005, 
        learning_rate='constant', 
        hidden_layer_sizes=(64, 32), 
        batch_size=128, 
        alpha=0.1, 
        activation='tanh'
    )

    weights= compute_sample_weight('balanced', y_tr)
    model.fit(X_tr, y_tr, sample_weight=weights)
    
    preds = model.predict(X_te)
    score = f1_score(y_te, preds, average='macro')
    scores_f1.append(score)
    
    print(f"Pli {fold + 1} - F1-Macro : {score}")

print(f"\n---> F1-Macro Moyen : {np.mean(scores_f1)}")

Pli 1 - F1-Macro : 0.3014555808227639
Pli 2 - F1-Macro : 0.2709956468187742
Pli 3 - F1-Macro : 0.29739322546073305
Pli 4 - F1-Macro : 0.30317113822550046
Pli 5 - F1-Macro : 0.23047001719171412

---> F1-Macro Moyen : 0.28069712170389716


In [ ]:
#entraînement de ce modèle sur tout le jeu de données d'entraînement

df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

X_train = df_x_clean.drop(columns=['id'])
y_train = df_y_clean['target']
X_test = df_x_test_clean.drop(columns=['id'])

model_final1 = MLPClassifier(
    random_state=42
)
weights= compute_sample_weight('balanced', y_train)

model_final1.fit(X_train, y_train, sample_weight=weights)

# prédiction sur le jeu de test
predictions = model_final1.predict(X_test)

# repasser des nombres aux noms de postes
postes = {
    0: "Assistant",
    1: "Executive",
    2: "Manager",
    3: "Director"
}
predictions_labels = [postes[p] for p in predictions]

# créer le fichier
predictions_final_1 = pd.DataFrame({
    'id': df_x_test_clean['id'],
    'position': predictions_labels
})
predictions_final_1.to_csv('../results/predictions_MLPClassifier_1.csv', index=True)


c:\Users\sarah\Documents\ENSC\3A\cours\ia\projet\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Le score de ces prédictions est **0,24424103799796493**

---
## Recherche hyperparamètre

In [ ]:
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

# les 32 colonnes d'entreprise
company_cols = [f'company_feature_{i}' for i in range(32)]

#regroupe sous un même identifiant les lignes qui ont la meme entreprise
df_x_clean['company_group'] = df_x_clean[company_cols].apply(lambda row: hash(tuple(row)), axis=1)
groups_company = df_x_clean['company_group']

#on va faire une validation croisée en utilisant StratifiedGroupKFold
#le découpage respecte à la fois les groupes d'entités et l'équilibre des classes dans chaque bloc.

X = df_x_clean.drop(columns=['id', 'company_group'])
y = df_y_clean['target']


# hyperparamètres à tester
param_test = {
    'hidden_layer_sizes': [(64,),(128,),(128, 64),(64, 32),(128, 64, 32)],
    'activation': ['relu', 'tanh'],
    'learning_rate_init': [0.0005, 0.001, 0.005, 0.01],
    'learning_rate': ['constant', 'adaptive'],
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.1],
    'batch_size': [32, 64, 128, 256],
    'max_iter': [300, 500]
}

sample_weights = compute_sample_weight('balanced', y)
model1 = MLPClassifier(
    random_state=42,
    early_stopping=True
)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# recherche aléatoire hyperparamètres
recherche = RandomizedSearchCV(
    estimator=model1,
    param_distributions=param_test,
    n_iter=20,
    scoring='f1_macro',
    cv=cv,
    random_state=42,
    n_jobs=1
)
recherche.fit(X, y, groups=groups_company, sample_weight=sample_weights)


print(f"Meilleur F1-Macro Réel: {recherche.best_score_:.4f}")
print("\nMeilleurs hyperparamètres trouvés :")
print(recherche.best_params_)

--- Meilleur F1-Macro Réel (GroupKFold) : 0.3513 ---

Meilleurs hyperparamètres trouvés :
{'max_iter': 300, 'learning_rate_init': 0.005, 'learning_rate': 'constant', 'hidden_layer_sizes': (64, 32), 'batch_size': 128, 'alpha': 0.1, 'activation': 'tanh'}


On va maintenant entrainer le même modèle avec les meilleurs hyperparamètres trouvés sur l'ensemble des données train 

Meilleurs hyperparamètres:  
max_iter: 300  
learning_rate_init: 0.005  
learning_rate: constant  
hidden_layer_sizes: (64, 32)  
batch_size: 128  
alpha: 0.1  
activation: tanh 


In [ ]:
#entraînement du modéle avec les meilleurs hyperparamètres sur tout le jeu de données d'entraînement

df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

X_train = df_x_clean.drop(columns=['id'])
y_train = df_y_clean['target']
X_test = df_x_test_clean.drop(columns=['id'])

model_final2 = MLPClassifier(
    random_state=42,
    early_stopping=True,
    max_iter=300, 
    learning_rate_init=0.005, 
    learning_rate='constant', 
    hidden_layer_sizes=(64, 32), 
    batch_size=128, 
    alpha=0.1, 
    activation='tanh'
)

model_final2.fit(X_train, y_train)

# prédiction sur le jeu de test
predictions = model_final2.predict(X_test)

# repasser des nombres aux noms de postes
postes = {
    0: "Assistant",
    1: "Executive",
    2: "Manager",
    3: "Director"
}
predictions_labels = [postes[p] for p in predictions]

# créer le fichier
predictions_final_2 = pd.DataFrame({
    'id': df_x_test_clean['id'],
    'position': predictions_labels
})
predictions_final_2.to_csv('../results/predictions_MLPClassifier_2.csv', index=True)

Le score de ces predictions est **0,24930570149068632**